# JiT-S2-VMamba2 — SS2D with a Mamba-2 (SSD) core

Our JiT-S2-VMamba baseline uses SS2D with the **Mamba-1** selective scan: one step $\Delta$ per channel (low-rank `dt_proj`) and one decay $A$ per (channel, state). This notebook swaps only the SSM core for **Mamba-2 (SSD)**, following VMamba's own Mamba-2 variant (`SS2Dm0`, `forward_type="m0_noz"`):

$$h_t = e^{\Delta_t^{(h)} A^{(h)}}\, h_{t-1} + \Delta_t^{(h)}\, x_t B_t^\top,\qquad y_t = h_t C_t + D \odot x_t,\qquad h_t \in \mathbb{R}^{16\times 16}\ \text{per head}$$

- **24 heads × headdim 16 per direction** (VMamba m0: `ceil(D/16)` heads); $\Delta$ and $A$ are one **per head**; B, C one group per direction.
- **d_state 16**: the same total state as the baseline (24 × 16 × 16 = 384 × 16 per direction).
- Mamba-2 module init ($A \sim U[1,16]$, $\Delta$ log-uniform in $[10^{-3}, 10^{-1}]$, the baseline's range); D per channel, as in the baseline.
- Everything else is the baseline SS2D: in_proj, DWConv 3×3 + SiLU, 4-direction cross-scan / merge, LayerNorm, out_proj, no gate. Every conditioning arm is off.
- Kernel: `mamba_chunk_scan_combined` (Triton, mamba-ssm 2.2.4).

`tests/test_vmamba2_cpu.py` proves this mixer **equals the Mamba-1 SS2D with $\Delta$ and $A$ tied per head** (float64), so the run isolates exactly that change.

| model | SSM core | params | GMACs |
|---|---|---|---|
| **JiT-VMamba2 (this)** | Mamba-2: Δ, A per head (24 × 16) | 30.28M | 1.393 |
| ref: JiT-VMamba baseline | Mamba-1: Δ per channel, A per channel × state | 31.03M | 1.421 |

GMACs: 64×64, patch 8, 64 tokens; scan counted as 9·L·D·N for both (the convention of `params_flops_tables.tex`).

**Triton on Kaggle.** Triton officially targets Ampere+ GPUs and a T4 has no bf16 tensor cores, so section 3b runs a probe that tries the kernel in bf16 and fp32 and sets `force_fp32` for you (or stops the notebook if the kernel cannot run on this GPU).

---

JiT-S2 on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The vmamba2 model, config and wiring are on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: the vmamba2 model is wired into both entry points
!grep -q 'arch == "vmamba2"' scripts/run_experiment.py && grep -q 'arch == "vmamba2"' scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify the model  *(first session only; seconds)*

In [ ]:
# === Verify the model (seconds; first session only -- comment out on resume).
# Builds the config exactly as evaluate.py will (on CPU) and checks it is the
# Mamba-2 SS2D: 24 heads x 16 per direction, Delta/A per head, D per channel,
# no dt_proj. If it FAILS, STOP -- do not train the wrong model.
import yaml
from scripts.evaluate import build_model
cfg = yaml.safe_load(open("configs/tiny_imagenet/jit-s2-vmamba2.yaml"))
m = build_model(cfg)
mx = m.blocks[0].mixer
n = sum(p.numel() for p in m.parameters())
checks = {
    "SS2DMamba2 mixer":         type(mx).__name__ == "SS2DMamba2",
    "24 heads x headdim 16":    (mx.nheads, mx.headdim) == (24, 16),
    "Delta, A per head":        tuple(mx.dt_bias.shape) == (4, 24) and tuple(mx.A_logs.shape) == (4, 24),
    "D per channel":            tuple(mx.Ds.shape) == (4, 384),
    "d_state 16, no dt_proj":   mx.d_state == 16 and not hasattr(mx, "dt_projs_weight"),
    "K = 4 directions":         mx.K == 4,
    "params 30,278,208":        n == 30_278_208,
}
for k, v in checks.items():
    print(f"{'OK  ' if v else 'FAIL'} {k}")
print(f"{n/1e6:.2f}M params")
assert all(checks.values()), "model check FAILED"
del m

## 3b. GPU probe  *(first session on a GPU type; ~1–2 min)*

In [ ]:
# === Does the Triton SSD kernel run on THIS GPU, and in which dtype? =========
# scripts/check_vmamba2_cuda.py compares the kernel against a float64 reference
# (fwd + bwd), runs the model end to end with force_fp32 false/true, times both
# against the Mamba-1 baseline, and ends with a VERDICT line. FORCE_FP32 is set
# from that verdict and written into the config in section 5.
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/check_vmamba2_cuda.py"],
                   capture_output=True, text=True, env={**os.environ, "PYTHONPATH": "."})
print(r.stdout[-6000:])
if r.returncode != 0 and r.stderr:
    print(r.stderr[-3000:])
verdict = next((l for l in r.stdout.splitlines() if l.startswith("VERDICT:")), "VERDICT: (none)")
if "force_fp32: false" in verdict:
    FORCE_FP32 = False
elif "force_fp32: true" in verdict:
    FORCE_FP32 = True
else:
    raise RuntimeError("The Triton SSD kernel does not run on this GPU -- "
                       "vmamba2 needs a PyTorch backend here. " + verdict)
print("-> FORCE_FP32 =", FORCE_FP32)

## 4. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 5. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

# Set by the probe (3b); if you skipped it on a resume, set it to the value the
# first session's probe printed.
try:
    FORCE_FP32
except NameError:
    FORCE_FP32 = False
    print("FORCE_FP32 not set by the probe; using:", FORCE_FP32)

CONFIG  = "configs/tiny_imagenet/jit-s2-vmamba2.yaml"
RUN     = "exp_vmamba2"
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "d_state": 16, "expand": 1, "K": 4,
                 "nheads": "null", "chunk_size": 64,
                 "force_fp32": str(FORCE_FP32).lower(), "ffn": "swiglu"})

# ── Resume across 12h sessions ─────────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_vmamba2".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "|", f"vmamba2 (Mamba-2 SS2D, 24x16 heads, d_state 16) force_fp32={FORCE_FP32}")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")

## 6. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the baseline, so the only variable is the SSM core.
- **This vs the baseline** = Mamba-1 → Mamba-2 in SS2D, i.e. per-channel Δ and per-(channel, state) A → one Δ and one A per head of 16 channels (proven to be the only difference by `tests/test_vmamba2_cpu.py` check C).
- −0.75M params vs the baseline (31.03M → 30.28M): no low-rank `dt_proj`, A per head instead of per channel × state. −2.0% GMACs (the `dt_proj` matmul); the scan is unchanged (same total state).
- `force_fp32: true` only changes the dtype the kernel computes in, not the model: a checkpoint trained with one setting evaluates with the other.
- Data lives in `/tmp` (not saved in the Version) to keep commits small; it re-downloads each session. To skip that, attach a Kaggle tiny-imagenet dataset and set `DATA_DIR` to it.